## This notebook provides the functionality to train FLASH on E5 cadets and generate data attributions. The later half of the notebook provide functions for extracting top k contributions and visualizing them. 

## You can choose to download the data yourself using the urls mentioned in this notebook or mount this notebook to the data directory in Beryl to run it automatically. The directory information is present in the repository Readme.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
import torch
from torch_geometric.data import Data
import os
import torch.nn.functional as F
import orjson as json
import warnings
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
warnings.filterwarnings('ignore')
from torch_geometric.loader import NeighborLoader
import gc

import multiprocessing

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
%matplotlib inline

In [ ]:
import networkx as nx
from pyvis.network import Network

In [ ]:
tokens = ['SUBJECT_PROCESS',
          'FILE_OBJECT_FILE',
          'NETFLOW']

df = pd.read_parquet('e5_data/cadets_df.parquet')
df['timestamp'] = pd.to_datetime(df['timestamp'])

start_range_end = '2019-05-9 00:00:00'
train_df = df[df['timestamp'] <= start_range_end]
train_df = train_df[train_df['object'].isin(tokens)]

second_range_start = '2019-05-16 00:00:00'
second_range_end = '2019-05-17 00:00:00'

test_df = df[(df['timestamp'] >= second_range_start) & (df['timestamp'] <= second_range_end)]
test_df = test_df[test_df['actor_type'] == 'SUBJECT_PROCESS'] 
test_df = test_df[test_df['object'].isin(tokens)]

In [ ]:
del df
gc.collect()

In [ ]:
len(train_df)

In [ ]:
len(set(train_df['exec']))

In [ ]:
Train = True

In [ ]:
from pprint import pprint
import gzip
from sklearn.manifold import TSNE
import json
import copy
import os

In [ ]:
'''
This is the main featurizer. It constructs the graph for the Cadets dataset.

Args:
    df (DataFrame): This is the main dataframe containing all the system events from the Cadets dataset.

return:
    features (list): Contains word2vec encoded feature vectors for each node
    feat_labels (list): Contains label for each node
    edge_index (list): Contains information about edges between nodes in the graph.
    mapp (list): contains id of each node
'''

tokens = ['SUBJECT_PROCESS',
          'FILE_OBJECT_FILE',
          'NETFLOW']

def prepare_graph(df):
    global tokens
    lblmap = {}

    dummies = {token: index for index, token in enumerate(tokens)}
    
    df['actor_label'] = df['actor_type'].map(dummies)
    df['object_label'] = df['object'].map(dummies)
    
    nodes = {}
    labels = {}
    for col in ['actorID', 'objectID']:
        unique_ids = df[col].unique()
        for uid in unique_ids:
            nodes[uid] = []
        if col == 'actorID':
            labels.update(df.set_index('actorID')['actor_label'].to_dict())
        else:
            labels.update(df.set_index('objectID')['object_label'].to_dict())
    
    for _, row in df.iterrows():
        nodes[row['actorID']].extend([row['exec'], row['action']])
        nodes[row['objectID']].extend([row['exec'], row['action']])
        if row['path'] != '':
            nodes[row['actorID']].append(row['path'])
            nodes[row['objectID']].append(row['path'])
            
        lblmap[row['actorID']] = row['exec']
        lblmap[row['objectID']] = row['path']
    
    edges = list(zip(df['actorID'], df['objectID']))

    mapp = list(nodes.keys())
    features = [nodes[node_id] for node_id in mapp]
    feat_labels = [labels[node_id] for node_id in mapp]
    edge_index = [[], []]
    index_map = {node_id: index for index, node_id in enumerate(mapp)}
    
    for src, dst in edges:
        edge_index[0].append(index_map[src])
        edge_index[1].append(index_map[dst])
    
    all_procs = list(df['actorID'].unique())
    idx_to_proc = {index: proc for index, proc in enumerate(all_procs)}

    return features, feat_labels, edge_index, mapp, lblmap

In [ ]:
from torch_geometric.nn import GCNConv
from torch_geometric.nn import SAGEConv, GATConv
import torch.nn.functional as F
import torch.nn as nn

class GCN(torch.nn.Module):
    def __init__(self,in_channel,out_channel):
        super().__init__()
        self.conv1 = SAGEConv(in_channel, 32, normalize=True)
        self.conv2 = SAGEConv(32, out_channel, normalize=True)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = x.relu()
        x = F.dropout(x, p=0.5, training=self.training)

        x = self.conv2(x, edge_index)
        return F.softmax(x, dim=1)

In [ ]:
def visualize(h, color):
    z = TSNE(n_components=2).fit_transform(h.detach().cpu().numpy())

    plt.figure(figsize=(10,10))
    plt.xticks([])
    plt.yticks([])

    plt.scatter(z[:, 0], z[:, 1], s=70, c=color, cmap="Set2")
    plt.show()

In [ ]:
from gensim.models.callbacks import CallbackAny2Vec
import gensim
from gensim.models import Word2Vec
from multiprocessing import Pool
from itertools import compress
from tqdm import tqdm
import time

class EpochSaver(CallbackAny2Vec):
    '''Callback to save model after each epoch.'''

    def __init__(self):
        self.epoch = 0

    def on_epoch_end(self, model):
        model.save('word2vec_cadets_E3.model')
        self.epoch += 1

In [ ]:
class EpochLogger(CallbackAny2Vec):
    '''Callback to log information about training'''

    def __init__(self):
        self.epoch = 0

    def on_epoch_begin(self, model):
        print("Epoch #{} start".format(self.epoch))

    def on_epoch_end(self, model):
        print("Epoch #{} end".format(self.epoch))
        self.epoch += 1

In [ ]:
logger = EpochLogger()
saver = EpochSaver()

In [ ]:
if Train:
    train_df.sort_values(by='timestamp', ascending=True,inplace=True)

In [ ]:
from sklearn.utils import class_weight
import torch.nn.functional as F
from torch.nn import CrossEntropyLoss

model = GCN(30,3).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)

In [ ]:
if Train:
    word2vec = Word2Vec(sentences=phrases, vector_size=30, window=5, min_count=1, workers=8,epochs=300,callbacks=[saver,logger])

In [ ]:
import math
import torch
import numpy as np
from gensim.models import Word2Vec

class PositionalEncoder:

    def __init__(self, d_model, max_len=100000):
        position = torch.arange(max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        self.pe = torch.zeros(max_len, d_model)
        self.pe[:, 0::2] = torch.sin(position * div_term)
        self.pe[:, 1::2] = torch.cos(position * div_term)

    def embed(self, x):
        return x + self.pe[:x.size(0)]


def infer(document):
    word_embeddings = [w2vmodel.wv[word] for word in document if word in  w2vmodel.wv]
    
    if not word_embeddings:
        return np.zeros(30)

    output_embedding = torch.tensor(word_embeddings, dtype=torch.float)
    if len(document) < 100000:
        output_embedding = encoder.embed(output_embedding)

    output_embedding = output_embedding.detach().cpu().numpy()
    return np.mean(output_embedding, axis=0)

encoder = PositionalEncoder(30)
w2vmodel = Word2Vec.load("Content_FL_Exp/E5_word2vec_Cadets_Flash.model")

In [ ]:
import os
import json
import pickle
import time

import torch
import numpy as np
from torch.nn import CrossEntropyLoss
from torch_geometric.data import Data
from torch_geometric.loader import NeighborLoader
from torch_geometric import utils
from sklearn.utils import class_weight

start = time.time()

candidates = list(set(train_df['exec']))

all_score_matrices = {}
target = 'zeroth'

while len(candidates) > 0:
    print("Target:", target)
    
    #---------------------------------------------------------------------------
    #                           TRAINING SECTION
    #---------------------------------------------------------------------------
    
    model = GCN(30,3).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)
    
    # Filter benign data to exclude the current target
    df_ben_trim = train_df[train_df["exec"] != target]

    # Prepare the graph
    phrases, labels, edges,mapp, _ = prepare_graph(df_ben_trim)
    
    # Infer embeddings for each node
    nodes = [infer(x) for x in phrases]
    nodes = np.array(nodes)

    # Define our training loss
    criterion = CrossEntropyLoss()

    # Create a torch_geometric graph
    graph = Data(
        x=torch.tensor(nodes, dtype=torch.float).to(device),
        y=torch.tensor(labels, dtype=torch.long).to(device),
        edge_index=torch.tensor(edges, dtype=torch.long).to(device)
    )

    for epochs in range(100):
        model.train()
        optimizer.zero_grad() 
        out = model(graph.x, graph.edge_index) 
        loss = criterion(out, graph.y) 
        loss.backward() 
        optimizer.step()      
        total_loss = loss.item() 
        print(total_loss)

    # Save model checkpoint
    torch.save(model.state_dict(), f'explain_lword2vec_gnn_cadets_E5.pth')
    
    #---------------------------------------------------------------------------
    #                           EVALUATION SECTION
    #---------------------------------------------------------------------------
    mode = "evaluation"
    
    # --- CACHING CHANGES ---
    eval_cache_file = "evaluation_cache_cadets_E5.pkl"
    if os.path.exists(eval_cache_file):
        # If cache file already exists, load from it
        with open(eval_cache_file, "rb") as f:
            eval_cache = pickle.load(f)

        all_ids = eval_cache["all_ids"]
        phrases = eval_cache["phrases"]
        labels = eval_cache["labels"]
        edges = eval_cache["edges"]
        mapp = eval_cache["mapp"]
        lbl = eval_cache["lbl"]
        nodes = eval_cache["nodes"]
        gt_nodes = eval_cache["gt_nodes"]

        print("Loaded evaluation data from cache.")
    else:
        # Otherwise, read raw data, transform, and create the graph
        phrases, labels, edges, mapp, lbl = prepare_graph(test_df)
        
        nodes = [infer(x) for x in phrases]
        nodes = np.array(nodes)  
        
        all_ids = list(test_df['actorID']) + list(test_df['objectID'])
        all_ids = set(all_ids)
        
        gt_nodes = set()
        for i in range(len(mapp)):
            if labels[i] == 0:
                gt_nodes.add(mapp[i])
        gt_nodes = list(gt_nodes)
        

        # Save data to cache
        eval_cache = {
            "all_ids": all_ids,
            "phrases": phrases,
            "labels": labels,
            "edges": edges,
            "mapp": mapp,
            "lbl": lbl,
            "nodes": nodes,
            "gt_nodes": gt_nodes,
        }
        with open(eval_cache_file, "wb") as f:
            pickle.dump(eval_cache, f)
        print("Evaluation data cached to disk.")

    graph = Data(x=torch.tensor(nodes,dtype=torch.float).to(device),y=torch.tensor(labels,dtype=torch.long).to(device), edge_index=torch.tensor(edges,dtype=torch.long).to(device))
    
    model.load_state_dict(torch.load(f'explain_lword2vec_gnn_cadets_E5.pth'))
    model.eval()
    out = model(graph.x, graph.edge_index)
    
    sorted, indices = out.sort(dim=1, descending=True)
    conf = (sorted[:,0] - sorted[:,1]) / sorted[:,0]
    conf = (conf - conf.min()) / conf.max()
    scores = conf
    
    idx_map = { v: i for i, v in enumerate(mapp) }
    score_matrix = {}
    
    for x in gt_nodes:
        ind = idx_map[x]                    
        score_matrix[lbl[x]] = float(scores[ind].item())

    # Store scores for this target in the global dictionary
    all_score_matrices[target] = score_matrix

    # Save all_score_matrices at the end of this iteration
    with open("all_score_matrices_cadets_E5.pkl", "wb") as f:
        pickle.dump(all_score_matrices, f)

    # Move to the next target
    target = candidates.pop()

print("Done with all targets. Final all_score_matrices saved.")

end = time.time()

In [ ]:
print('Total_time =', end - start)

In [ ]:
def find_top_k_contributing_samples(all_score_matrices, k=3):
    """
    For each malicious process, find the top k omitted training processes 
    that cause the largest *increase* in anomaly score relative to baseline.

    Returns a dictionary of:
      malicious_process -> list of tuples (omitted_sample, score_difference),
    where the list is sorted by score_difference in descending order.
    """
    # Baseline scores (no training sample omitted)
    baseline_scores = all_score_matrices['zeroth']

    # This will store: malicious_process -> list of (omitted_sample, diff_from_baseline)
    top_k_contributions = {}

    # Loop through each malicious process in the baseline
    for malicious_proc, baseline_score in baseline_scores.items():
        # We'll accumulate (omitted_sample, diff) for all omitted samples
        diff_list = []

        # Compare scores from each omitted-sample scenario
        for omitted_sample, score_dict in all_score_matrices.items():
            # Skip the baseline key itself
            if omitted_sample == 'zeroth':
                continue

            # Get the anomaly score for the malicious process under this omission
            current_score = score_dict.get(malicious_proc)
            if current_score is None:
                # If for some reason this malicious_proc wasn't evaluated, skip
                continue

            # Calculate how much the anomaly score differs from the baseline
            diff_from_baseline = current_score - baseline_score

            diff_list.append((omitted_sample, diff_from_baseline))

        # Sort by difference in descending order (largest increase first)
        diff_list.sort(key=lambda x: x[1], reverse=True)

        # Take the top k
        top_k_contributions[malicious_proc] = diff_list[:k]

    return top_k_contributions

In [ ]:
import pickle

with open("all_score_matrices_cadets_E5.pkl", "rb") as f:
    all_score_matrices = pickle.load(f)

# Attribution Results

In [ ]:
results = find_top_k_contributing_samples(all_score_matrices, k=3)
for malicious_proc, top_list in results.items():
    print(f"Malicious process: {malicious_proc}")
    for (omitted_sample, diff) in top_list:
        print(f"  Omitted sample: {omitted_sample}, Score difference: {diff:.4f}")
    print()

In [ ]:
raw_edges = []
df = train_df
for _, row in df.iterrows():
    raw_props = row.get("properties", {})
    
    if not raw_props:
        continue

    actorid = row.get("exec")
    objectid = raw_props.get("objectpath") if raw_props.get("objectpath") else raw_props.get("remoteAddress", '')
    raw_edges.append([actorid, objectid])

In [ ]:
raw_edges = df[['exec', 'path']].values.tolist()

In [ ]:
import pandas as pd
import networkx as nx
from pyvis.network import Network

from collections import deque, defaultdict

def k_hop_subgraph(edges, target_node, k):
    adj = defaultdict(list)
    for u, v in edges:
        adj[u].append(v)
        adj[v].append(u)

    visited = {target_node}
    queue = deque([(target_node, 0)]) 
    while queue:
        node, depth = queue.popleft()
        if depth == k:
            continue
        for nbr in adj[node]:
            if nbr not in visited:
                visited.add(nbr)
                queue.append((nbr, depth + 1))

    sub_edges = [(u, v) for u, v in edges if u in visited and v in visited]
    return sub_edges

In [ ]:
target_node = "expr"  
edges = k_hop_subgraph(raw_edges, target_node, 2)

In [ ]:
print(len(edges))

In [ ]:
import networkx as nx
from pyvis.network import Network

# Build your main graph
G = nx.Graph()
G.add_edges_from(edges)

# Remove self loops (if any)
self_loops = list(nx.selfloop_edges(G))
G.remove_edges_from(self_loops)

# `nx.single_source_shortest_path_length` returns a dictionary: {node: distance}
distances = nx.single_source_shortest_path_length(G, target_node, cutoff=4)
# Extract all nodes (keys) found within distance <= 2
two_hop_nodes = list(distances.keys())

# Create a subgraph with only those nodes
H = G.subgraph(two_hop_nodes).copy()

# --- Now build a PyVis network from this subgraph ---
net = Network(
    notebook=True,
    height="750px",
    width="100%",
    bgcolor="#222222",
    font_color="white",
    cdn_resources='in_line'
)

# Add nodes with color customization
for node in H.nodes():
    if node == target_node:
        net.add_node(
            node,
            title=node,
            label=node,
            color="red"   # Highlight the target node in red
        )
    else:
        net.add_node(
            node,
            title=node,
            label=node,
            color="green" # Default color for non-target nodes
        )

# Add edges with color customization
for source, target in H.edges():
    # If the edge involves the target node, color it red
    if source == target_node or target == target_node:
        net.add_edge(source, target, color="red")
    else:
        net.add_edge(source, target, color="gray")

# (Optional) Customize layout/physics
net.set_options("""
var options = {
    "physics": {
        "forceAtlas2Based": {
            "gravitationalConstant": -26,
            "centralGravity": 0.005,
            "springLength": 230,
            "springConstant": 0.18
        },
        "maxVelocity": 146,
        "solver": "forceAtlas2Based",
        "timestep": 0.35,
        "stabilization": {"iterations": 150}
    },
    "nodes": {
        "font": {
            "size": 12
        }
    }
}
""")

# Show or save
net.show("graph_explanation.html")